## Model Packaging

In [ ]:
import os
import sys
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
import joblib
from river import drift

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── File paths ────────────────────────────────────────────────────────────────
MODELS_DIR      = "models"
REPORTS_DIR     = os.path.join("reports", "figures")
DATA_SAMPLE_DIR = os.path.join("data", "sample")

MODEL_META    = os.path.join(MODELS_DIR, "model_meta.joblib")
XGB_SPW       = os.path.join(MODELS_DIR, "xgb_spw.joblib")
TRAIN_PARQUET = os.path.join("data", "train.parquet")
TEST_PARQUET  = os.path.join("data", "test.parquet")

BUNDLE_PATH       = os.path.join(MODELS_DIR, "streamlit_bundle.joblib")
DEMO_DATA_PATH    = os.path.join(DATA_SAMPLE_DIR, "demo_transactions.parquet")
APP_PY_PATH       = "app.py"
REQUIREMENTS_PATH = "requirements.txt"

# ── Drift-stream replay config ─────────────────────────────────────────────────
# These MUST match Stage 12 exactly so the Streamlit chart is consistent with
# the notebook visualization. If Stage 12 used different values, update here.
STREAM_SIZE       = 20_000   # rows of test data to replay in the demo stream
ROLLING_WINDOW    = 200      # rolling window for the error-rate line chart
ADWIN_DELTA       = 0.002    # ADWIN sensitivity (matches Stage 12)
DRIFT_INJECT_FRAC = 0.70     # inject drift at the 70% mark of the stream

# ── Demo dataset config ────────────────────────────────────────────────────────
DEMO_FRAUD_N = 100    # fraud rows in the pre-scored demo file
DEMO_LEGIT_N = 400    # legitimate rows in the pre-scored demo file

print(" Imports and configuration ready.")
print(f"   Python      : {sys.version.split()[0]}")
print(f"   scikit-learn: {sklearn.__version__}")
print(f"   XGBoost     : {xgb.__version__}")

In [ ]:
REQUIRED_FILES = {
    "model_meta.joblib" : MODEL_META,
    "xgb_spw.joblib"    : XGB_SPW,
    "data/train.parquet": TRAIN_PARQUET,
    "data/test.parquet" : TEST_PARQUET,
}

# Figures are loaded by the Streamlit app; missing ones are handled gracefully
# in the app with os.path.exists() checks, so we warn here instead of failing.
EXPECTED_FIGURES = [
    "pr_curves_comparison.png",
    "roc_curves_comparison.png",
    "confusion_matrices_default_threshold.png",
    "shap_bar_chart.png",
]

print("Checking prerequisites...\n")
all_ok = True

for label, path in REQUIRED_FILES.items():
    exists = os.path.exists(path)
    size   = f"({os.path.getsize(path) / 1024**2:.1f} MB)" if exists else ""
    status = "EXISTS" if exists else "MISSING"
    print(f"  {status}  {label:<38s} {size}")
    if not exists:
        all_ok = False

print()
for fname in EXPECTED_FIGURES:
    path   = os.path.join(REPORTS_DIR, fname)
    exists = os.path.exists(path)
    status = "EXISTS" if exists else "  not found (app will skip this image)"
    print(f"  {status}  reports/figures/{fname}")

print()
assert all_ok, (
    "\n One or more required files are missing.\n"
    "   Complete all previous stages before running Stage 13."
)
print(" All required files present. Proceeding.\n")

In [ ]:
# =============================================================================
# %% CELL 4 — Load All Artifacts and Smoke-Test the Model
# =============================================================================
# Load model_meta (the accumulating dict from Stages 8–12) and the primary
# model. Then run a sanity-check prediction to confirm the pipeline still
# works end-to-end before we pack anything into the bundle.
#
# Defensive key extraction:
#   model_meta was written across multiple stages by different cells. Key names
#   may vary slightly. _get() tries a list of candidate names and returns the
#   first match, falling back to a default rather than crashing.

meta          = joblib.load(MODEL_META)
model_xgb_spw = joblib.load(XGB_SPW)

def _get(d, *keys, default=0.0):
    """Return the first matching key from a dict, or a default."""
    for k in keys:
        if k in d:
            return d[k]
    return default

# Core pipeline objects
FEATURE_COLS       = meta["feature_cols"]
DECISION_THRESHOLD = _get(meta, "DECISION_THRESHOLD", default=0.5)
FN_COST            = _get(meta, "FN_COST",            default=500)
FP_COST            = _get(meta, "FP_COST",            default=10)

# Explainability (Stage 11)
TOP_SHAP_FEATURE   = _get(meta, "TOP_SHAP_FEATURE",       default=FEATURE_COLS[0])
SHAP_RANKING       = _get(meta, "SHAP_FEATURE_RANKING",   default=FEATURE_COLS)
SHAP_MEAN_ABS      = _get(meta, "SHAP_MEAN_ABS_VALUES",   default={})

# Performance metrics — try the key names Stage 9–10 most likely produced
METRICS = {
    "pr_auc"        : _get(meta, "PR_AUC_XGB_SPW",   "PR_AUC",
                            "xgb_best_val_aucpr"),
    "roc_auc"       : _get(meta, "ROC_AUC_XGB_SPW",  "ROC_AUC"),
    "f1"            : _get(meta, "F1_TUNED",          "F1_SCORE"),
    "precision"     : _get(meta, "PRECISION_TUNED",   "PRECISION"),
    "recall"        : _get(meta, "RECALL_TUNED",      "RECALL"),
    "fraud_rate_pct": 0.129,
}

print("Loaded artifacts:")
print(f"  DECISION_THRESHOLD : {DECISION_THRESHOLD:.4f}")
print(f"  FN_COST / FP_COST  : ${int(FN_COST)} / ${int(FP_COST)}")
print(f"  TOP_SHAP_FEATURE   : {TOP_SHAP_FEATURE}")
print(f"  Metrics extracted  : {METRICS}")
print()

# Print actual feature column names — critical for understanding
# what your Stage 5 produced and what the app.py will need to match
print(f"FEATURE_COLS ({len(FEATURE_COLS)} features):")
for i, f in enumerate(FEATURE_COLS):
    print(f"  [{i}]  {f}")
print()

# ── Smoke-test: predict on actual rows from the test set ─────────────────────
# Using real rows (not hardcoded feature values) makes the smoke-test
# independent of whatever naming convention Stage 5 used.
# This is also a better test: real data will surface NaN / dtype issues that
# hand-crafted dicts might silently avoid.

print(" Loading two test rows for smoke-test (fraud + legit)...")
_df_st    = pd.read_parquet(TEST_PARQUET)
_fraud_ex = _df_st[_df_st["isFraud"] == 1][FEATURE_COLS].head(1)
_legit_ex = _df_st[_df_st["isFraud"] == 0][FEATURE_COLS].head(1)
del _df_st   # free RAM — Cell 6 reloads the full test set

assert len(_fraud_ex) == 1, "No fraud rows found in test set — check TEST_PARQUET."
assert len(_legit_ex) == 1, "No legit rows found in test set — check TEST_PARQUET."

print("\nSmoke-test predictions:")
for label, X_row in [
    ("Confirmed fraud row from test set", _fraud_ex),
    ("Confirmed legit row from test set", _legit_ex),
]:
    prob = float(model_xgb_spw.predict_proba(X_row)[0, 1])
    flag = prob >= DECISION_THRESHOLD
    result = " FLAGGED" if flag else " CLEARED"
    print(f"  {label:<40s}  P(fraud) = {prob:.4f}  →  {result}")

print()
print("✅ Smoke-test passed — model loads, feature columns align, predictions run.")

In [ ]:
print(" Loading training data for feature statistics...")

df_train = pd.read_parquet(TRAIN_PARQUET)
print(f"  Training rows : {len(df_train):,}")
print(f"  Fraud in train: {df_train['isFraud'].sum():,} "
      f"({df_train['isFraud'].mean()*100:.4f}%)")

legit_mask = df_train["isFraud"] == 0
fraud_mask = df_train["isFraud"] == 1

train_stats = {
    "legit_mean": df_train.loc[legit_mask, FEATURE_COLS].mean().to_dict(),
    "fraud_mean": df_train.loc[fraud_mask, FEATURE_COLS].mean().to_dict(),
}

print("\nFeature means — Legitimate vs. Fraud:\n")
print(f"  {'Feature':<30s}  {'Legit Mean':>15s}  {'Fraud Mean':>15s}  {'Ratio':>8s}")
print("  " + "─" * 74)
for feat in FEATURE_COLS:
    lm    = train_stats["legit_mean"][feat]
    fm    = train_stats["fraud_mean"][feat]
    ratio = (fm / lm) if abs(lm) > 1e-9 else float("nan")
    r_str = f"{ratio:>8.2f}x" if not np.isnan(ratio) else "     N/A"
    print(f"  {feat:<30s}  {lm:>15.4f}  {fm:>15.4f}  {r_str}")

print()
print("  → Large ratios confirm that errorBalanceOrig and errorBalanceDest")
print("    are the dominant fraud signals — matching the SHAP global ranking.")

del df_train   # free RAM before loading test data
print("\n Training statistics computed and stored in train_stats.")

In [ ]:
print("⏳ Loading test data...")
df_test = pd.read_parquet(TEST_PARQUET)
print(f"  Test rows : {len(df_test):,}")
print(f"  Fraud     : {df_test['isFraud'].sum():,} "
      f"({df_test['isFraud'].mean()*100:.4f}%)")

os.makedirs(DATA_SAMPLE_DIR, exist_ok=True)

fraud_pool = df_test[df_test["isFraud"] == 1]
legit_pool = df_test[df_test["isFraud"] == 0]

n_fraud = min(DEMO_FRAUD_N, len(fraud_pool))
n_legit = min(DEMO_LEGIT_N, len(legit_pool))

demo_df = pd.concat([
    fraud_pool.sample(n_fraud, random_state=RANDOM_STATE),
    legit_pool.sample(n_legit, random_state=RANDOM_STATE),
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Add the model's fraud probability and binary flag to each demo row.
# The app can use these for a pre-scored results table in Tab 1.
X_demo             = demo_df[FEATURE_COLS]
demo_df["fraud_prob"] = model_xgb_spw.predict_proba(X_demo)[:, 1]
demo_df["flagged"]    = (demo_df["fraud_prob"] >= DECISION_THRESHOLD).astype(int)

demo_df.to_parquet(DEMO_DATA_PATH, index=False)
size_kb = os.path.getsize(DEMO_DATA_PATH) / 1024

print(f"\n  Demo dataset : {len(demo_df)} rows  "
      f"({n_fraud} fraud + {n_legit} legitimate)")
print(f"  Saved to     : {DEMO_DATA_PATH}  ({size_kb:.1f} KB)")
print()
print("  Score preview (5 rows):")
cols_to_show = ["isFraud", "fraud_prob", "flagged", "amount"]
print(demo_df[cols_to_show].head().to_string(index=False))
print()

# Quick accuracy check on the demo sample (should be high given it's the best model)
correct = (demo_df["flagged"] == demo_df["isFraud"]).sum()
print(f"  Demo accuracy : {correct}/{len(demo_df)} "
      f"({correct/len(demo_df)*100:.1f}%)")

In [ ]:
# ── TASK A: Rebuild the ADWIN stream ─────────────────────────────────────────
print(" Rebuilding ADWIN drift stream...")
print(f"  Stream size       : {STREAM_SIZE:,} rows")
print(f"  Drift injected at : {DRIFT_INJECT_FRAC:.0%} mark")
print(f"  ADWIN delta       : {ADWIN_DELTA}")
print(f"  TOP_SHAP_FEATURE  : {TOP_SHAP_FEATURE}")
print()

# ── Temporal ordering ─────────────────────────────────────────────────────────


if "step" in df_test.columns:
    print("  'step' column found — sorting stream by simulation time.")
    df_stream = (
        df_test
        .sort_values("step")
        .reset_index(drop=True)
        .head(STREAM_SIZE)
        .copy()
    )
else:
    print("  'step' not in test.parquet — using saved row order.")
    print("  (Stage 6 time-aware split already places test rows in later steps.)")
    df_stream = df_test.head(STREAM_SIZE).reset_index(drop=True).copy()

# ── Sanity checks before drift injection ─────────────────────────────────────
assert "isFraud" in df_stream.columns, (
    " 'isFraud' not found in test.parquet. "
    "Check that Stage 6 saved the target column."
)
assert TOP_SHAP_FEATURE in df_stream.columns, (
    f" TOP_SHAP_FEATURE '{TOP_SHAP_FEATURE}' not in test.parquet columns.\n"
    f"   Available columns: {list(df_stream.columns)}\n"
    f"   Check that FEATURE_COLS in model_meta match the saved parquet."
)

n_fraud_in_stream = int(df_stream["isFraud"].sum())
print(f"\n  Stream shape      : {df_stream.shape}")
print(f"  Fraud in stream   : {n_fraud_in_stream:,} "
      f"({n_fraud_in_stream/len(df_stream)*100:.3f}%)")

# ── Inject drift ──────────────────────────────────────────────────────────────
# Negate TOP_SHAP_FEATURE for fraud rows AFTER the 70% mark.
# This mirrors Stage 12 exactly. The model's strongest fraud signal is
# flipped so transactions it learned to flag now look legitimate to it.
inject_idx = int(len(df_stream) * DRIFT_INJECT_FRAC)

df_drifted = df_stream.copy()
drift_mask = (df_drifted.index >= inject_idx) & (df_drifted["isFraud"] == 1)
n_affected = int(drift_mask.sum())

df_drifted.loc[drift_mask, TOP_SHAP_FEATURE] = (
    -df_drifted.loc[drift_mask, TOP_SHAP_FEATURE]
)

print(f"  Drift injected at : index {inject_idx:,} "
      f"({DRIFT_INJECT_FRAC:.0%} mark)")
print(f"  Rows affected     : {n_affected:,} fraud rows negated")

# ── Stream through ADWIN, collect binary errors ───────────────────────────────
adwin               = drift.ADWIN(delta=ADWIN_DELTA)
errors              = []
drift_event_indices = []

X_stream = df_drifted[FEATURE_COLS].values
y_stream = df_drifted["isFraud"].values

for i in range(len(X_stream)):
    prob_i = float(model_xgb_spw.predict_proba(X_stream[i : i + 1])[0, 1])
    pred_i = int(prob_i >= DECISION_THRESHOLD)
    err_i  = int(pred_i != int(y_stream[i]))   # 1 = wrong, 0 = correct
    errors.append(err_i)
    adwin.update(err_i)
    if adwin.drift_detected:
        drift_event_indices.append(i)

print(f"\n  Stream processed  : {len(errors):,} transactions")
print(f"  Overall error rate: {np.mean(errors)*100:.2f}%")
print(f"  Drift events found: {drift_event_indices}")

if not drift_event_indices:
    print()
    print("    No drift events detected. This can happen if:")
    print("     • The ADWIN window hasn't accumulated enough signal yet")
    print("       (try lowering ADWIN_DELTA in Cell 1, e.g. 0.02).")
    print("     • The injected feature has low variance in fraud rows")
    print("       after the 70% mark (check TOP_SHAP_FEATURE values).")
    print("     • Very few fraud rows exist past the inject_idx.")
    print(f"     Fraud rows after inject_idx: {n_affected}")

# ── Build rolling error DataFrame for the Plotly chart ───────────────────────
errors_arr  = np.array(errors, dtype=float)
rolling_err = (
    pd.Series(errors_arr)
    .rolling(window=ROLLING_WINDOW, min_periods=1)
    .mean()
    .values
)

drift_stream_df = pd.DataFrame({
    "idx"          : np.arange(len(errors)),
    "error"        : errors,
    "rolling_error": rolling_err,
})

# ── TASK B: Assemble and save the bundle ─────────────────────────────────────
bundle = {
    # Core model artifacts
    "model"              : model_xgb_spw,
    "feature_cols"       : FEATURE_COLS,
    "threshold"          : DECISION_THRESHOLD,
    "fn_cost"            : FN_COST,
    "fp_cost"            : FP_COST,
    # Performance metrics (Stage 9–10)
    "metrics"            : METRICS,
    # Explainability (Stage 11)
    "shap_ranking"       : SHAP_RANKING,
    "shap_mean_abs"      : SHAP_MEAN_ABS,
    "top_shap_feature"   : TOP_SHAP_FEATURE,
    # Training distribution reference for Tab 1 feature chart
    "train_stats"        : train_stats,
    # Drift monitoring
    "drift_events"       : drift_event_indices,
    "drift_inject_index" : inject_idx,
    "drift_stream_df"    : drift_stream_df,
    # Provenance
    "created_at"         : datetime.now().isoformat(),
    "python_version"     : sys.version.split()[0],
    "sklearn_version"    : sklearn.__version__,
    "xgboost_version"    : xgb.__version__,
}

os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump(bundle, BUNDLE_PATH, compress=3)

size_mb = os.path.getsize(BUNDLE_PATH) / 1024 ** 2
print(f"\n Bundle saved: {BUNDLE_PATH}  ({size_mb:.2f} MB)")
print(f"   Keys: {list(bundle.keys())}")

In [ ]:
# =============================================================================
# %% CELL 8 — Bundle Verification + Stage 13 Checkpoint
# =============================================================================


print(" Verifying bundle (reloading from disk)...\n")
b = joblib.load(BUNDLE_PATH)

REQUIRED_BUNDLE_KEYS = [
    "model", "feature_cols", "threshold", "metrics",
    "shap_ranking", "shap_mean_abs", "train_stats",
    "drift_events", "drift_stream_df", "drift_inject_index",
    "created_at", "python_version", "sklearn_version", "xgboost_version",
]
missing = [k for k in REQUIRED_BUNDLE_KEYS if k not in b]
assert not missing, f" Bundle is missing keys: {missing}"

# ── End-to-end prediction using ONLY bundle objects ───────────────────────────
# Pull one fraud row and one legit row from the demo dataset created in Cell 6.
# demo_df already has FEATURE_COLS columns and isFraud, so it's the ideal
# test fixture — real data, real column names, no hardcoding.
assert os.path.exists(DEMO_DATA_PATH), (
    f" Demo dataset not found at {DEMO_DATA_PATH}. "
    "Did Cell 6 complete successfully?"
)

_demo = pd.read_parquet(DEMO_DATA_PATH)
_fraud_row = _demo[_demo["isFraud"] == 1][b["feature_cols"]].head(1)
_legit_row = _demo[_demo["isFraud"] == 0][b["feature_cols"]].head(1)

assert len(_fraud_row) == 1, "No fraud rows in demo dataset."
assert len(_legit_row) == 1, "No legit rows in demo dataset."

print("End-to-end predictions using reloaded bundle:")
for label, X_row in [
    ("Confirmed fraud row", _fraud_row),
    ("Confirmed legit row", _legit_row),
]:
    _prob = float(b["model"].predict_proba(X_row)[0, 1])
    _flag = _prob >= b["threshold"]
    result = " FLAGGED" if _flag else " CLEARED"
    print(f"  {label:<25s}  P(fraud) = {_prob:.4f}  →  {result}")

del _demo, _fraud_row, _legit_row

# ── Bundle summary ────────────────────────────────────────────────────────────
print()
print(f"  Model object       : {type(b['model']).__name__}")
print(f"  Feature cols       : {b['feature_cols']}")
print(f"  Threshold          : {b['threshold']:.4f}")
print(f"  Drift events       : {b['drift_events']}")
print(f"  Drift stream rows  : {len(b['drift_stream_df']):,}")
print(f"  Created at         : {b['created_at']}")
print(f"  Bundle on disk     : {os.path.getsize(BUNDLE_PATH)/1024**2:.2f} MB")
del b

print()
print("=" * 62)
print("  STAGE 13 COMPLETE — DEPLOYMENT BUNDLE SUMMARY")
print("=" * 62)
print()
print(f"  Bundle path        : {BUNDLE_PATH}")
print(f"  Demo dataset       : {DEMO_DATA_PATH}")
print(f"  Model              : XGBClassifier (SPW)")
print(f"  Feature order      : {FEATURE_COLS}")
print(f"  Decision threshold : {DECISION_THRESHOLD:.4f}  "
      f"(tuned: FN=${int(FN_COST)} / FP=${int(FP_COST)})")
print(f"  PR-AUC             : {METRICS['pr_auc']:.4f}")
print(f"  Recall             : {METRICS['recall']:.4f}")
print(f"  Top SHAP feature   : {TOP_SHAP_FEATURE}")
print(f"  Drift events       : {drift_event_indices}")
print()
print("  NEXT: Stage 14 — Streamlit App Design & Export")
print("=" * 62)

## Streamlit App Design

In [ ]:
# =============================================================================
# %% CELL 10 — Write app.py to Disk
# =============================================================================


APP_CODE = """# app.py — PaySim Fraud Detector
# Generated by fraud_detection_stage13_14.ipynb  (Stage 14)
# ─────────────────────────────────────────────────────────────────────────────
# Run locally  :  streamlit run app.py
# Pre-requisites (relative to repo root):
#   models/streamlit_bundle.joblib
#   data/sample/demo_transactions.parquet
#   reports/figures/*.png   (optional — app skips missing images gracefully)
# ─────────────────────────────────────────────────────────────────────────────
# Feature set (11 features, must match training order exactly):
#   amount, type_TRANSFER,
#   oldbalanceOrg, newbalanceOrig, oldbalanceDest, newbalanceDest,
#   errorBalanceOrig, errorBalanceDest,
#   flag_orig_zero_after, flag_dest_zero_before, flag_dest_zero_both

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import joblib
import os

# ── Page configuration ────────────────────────────────────────────────────────
st.set_page_config(
    page_title='PaySim Fraud Detector',
    page_icon='\\U0001F6A8',
    layout='wide',
    initial_sidebar_state='expanded',
)

BUNDLE_PATH    = 'models/streamlit_bundle.joblib'
DEMO_DATA_PATH = 'data/sample/demo_transactions.parquet'

# ── Caching ───────────────────────────────────────────────────────────────────
# cache_resource : load once, share across ALL users and reruns (read-only model).
# cache_data     : return a copy per caller (mutable DataFrames).

@st.cache_resource
def load_bundle():
    if not os.path.exists(BUNDLE_PATH):
        st.error('Bundle not found: ' + BUNDLE_PATH)
        st.error('Run fraud_detection_stage13_14.ipynb (Stage 13) first.')
        st.stop()
    return joblib.load(BUNDLE_PATH)

@st.cache_data
def load_demo():
    if os.path.exists(DEMO_DATA_PATH):
        return pd.read_parquet(DEMO_DATA_PATH)
    return None

bundle  = load_bundle()
demo_df = load_demo()

model        = bundle['model']
FEATURE_COLS = bundle['feature_cols']
THRESHOLD    = bundle['threshold']
metrics      = bundle.get('metrics', {})
train_stats  = bundle.get('train_stats', {})
shap_ranking = bundle.get('shap_ranking', [])
shap_mean_abs= bundle.get('shap_mean_abs', {})
drift_df     = bundle.get('drift_stream_df')
drift_events = bundle.get('drift_events', [])
inject_idx   = bundle.get('drift_inject_index')

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.title('\\U0001F6A8 PaySim Fraud Detector')
    st.caption('XGBoost + ADWIN Drift Monitor')
    st.divider()
    st.markdown('**Model snapshot**')
    st.metric(
        'PR-AUC', f"{metrics.get('pr_auc', 0):.3f}",
        help='Primary metric — not inflated by true negatives',
    )
    st.metric(
        'Recall', f"{metrics.get('recall', 0):.1%}",
        help='% of all real frauds caught at the tuned threshold',
    )
    st.metric(
        'Threshold', f'{THRESHOLD:.3f}',
        help='Cost-tuned: FN=$500 / FP=$10 (Stage 10)',
    )
    st.divider()
    st.caption('Created  : ' + bundle.get('created_at', 'N/A')[:10])
    st.caption('XGBoost  : ' + bundle.get('xgboost_version', 'N/A'))
    st.caption('sklearn  : ' + bundle.get('sklearn_version', 'N/A'))
    st.divider()
    st.markdown('**Feature set (' + str(len(FEATURE_COLS)) + ' features)**')
    for f in FEATURE_COLS:
        st.caption('• ' + f)

# ── Tabs ──────────────────────────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs([
    '\\U0001F50D Score a Transaction',
    '\\U0001F4CA Model Performance',
    '\\U0001F4E1 Drift Monitor',
])

# ==========================================================================
# TAB 1 — Live Transaction Scorer
# ==========================================================================
with tab1:
    st.header('Score a Transaction')
    st.markdown(
        'Enter the six raw transaction fields. All engineered features '
        '(balance errors, zero-balance flags, transaction type flag) '
        'are computed automatically before the model scores the transaction.'
    )

    # Default values pre-set to a classic PaySim fraud signature:
    #   Large TRANSFER, origin drained beyond the transfer amount,
    #   destination balance zero both before and after.
    #   This means on first click the form should return a FRAUD flag.
    with st.form('score_form'):
        c1, c2 = st.columns(2)
        with c1:
            txn_type = st.selectbox(
                'Transaction Type', ['TRANSFER', 'CASH_OUT'],
                help='Fraud only occurs in TRANSFER and CASH_OUT transactions.',
            )
            amount = st.number_input(
                'Amount', min_value=0.0, value=55_000.0, step=1_000.0,
            )
        with c2:
            old_bal_orig = st.number_input(
                'Origin Balance BEFORE',
                min_value=0.0, value=100_000.0, step=1_000.0,
                help='oldbalanceOrg — sender account balance before transfer',
            )
            new_bal_orig = st.number_input(
                'Origin Balance AFTER',
                min_value=0.0, value=0.0, step=1_000.0,
                help='newbalanceOrig — sender account balance after transfer',
            )
            old_bal_dest = st.number_input(
                'Destination Balance BEFORE',
                min_value=0.0, value=0.0, step=1_000.0,
                help='oldbalanceDest — receiver account balance before transfer',
            )
            new_bal_dest = st.number_input(
                'Destination Balance AFTER',
                min_value=0.0, value=0.0, step=1_000.0,
                help='newbalanceDest — receiver account balance after transfer',
            )

        submitted = st.form_submit_button(
            '\\U0001F50D Score Transaction', use_container_width=True,
        )

    if submitted:
        # ── Compute all engineered features ──────────────────────────────────
        # These must exactly mirror Stage 5's feature engineering pipeline.
        #
        # errorBalanceOrig: money the origin lost beyond the stated amount.
        #   For a legitimate transfer: (oldbalanceOrg - amount) == newbalanceOrig
        #   so errorBalanceOrig == 0.  A large negative value means the origin
        #   lost more than it should have.
        #
        # errorBalanceDest: money the destination should have gained but did not.
        #   For a legitimate transfer: (oldbalanceDest + amount) == newbalanceDest
        #   so errorBalanceDest == 0.  A large positive value means the destination
        #   never received the money — a hallmark of PaySim fraud.
        #
        # flag_orig_zero_after  : origin account drained to exactly zero.
        # flag_dest_zero_before : destination account was empty before receipt.
        # flag_dest_zero_both   : destination was empty both before and after
        #                         (money passed straight through or never arrived).

        err_orig           = float(new_bal_orig + amount - old_bal_orig)
        err_dest           = float(old_bal_dest + amount - new_bal_dest)
        flag_orig_zero_after   = int(new_bal_orig == 0.0)
        flag_dest_zero_before  = int(old_bal_dest == 0.0)
        flag_dest_zero_both    = int(old_bal_dest == 0.0 and new_bal_dest == 0.0)
        type_transfer          = int(txn_type == 'TRANSFER')

        # Build feature row — order enforced by bundle's feature_cols list.
        # Never rely on dict insertion order: always reindex with FEATURE_COLS.
        row = {
            'amount'               : float(amount),
            'type_TRANSFER'        : type_transfer,
            'oldbalanceOrg'        : float(old_bal_orig),
            'newbalanceOrig'       : float(new_bal_orig),
            'oldbalanceDest'       : float(old_bal_dest),
            'newbalanceDest'       : float(new_bal_dest),
            'errorBalanceOrig'     : err_orig,
            'errorBalanceDest'     : err_dest,
            'flag_orig_zero_after' : flag_orig_zero_after,
            'flag_dest_zero_before': flag_dest_zero_before,
            'flag_dest_zero_both'  : flag_dest_zero_both,
        }

        X    = pd.DataFrame([row])[FEATURE_COLS]   # enforces correct column order
        prob = float(model.predict_proba(X)[0, 1])
        flag = prob >= THRESHOLD

        # ── Results ──────────────────────────────────────────────────────────
        st.divider()
        mc1, mc2, mc3 = st.columns(3)
        mc1.metric('Fraud Probability', f'{prob:.1%}')
        mc2.metric('Decision Threshold', f'{THRESHOLD:.3f}')
        mc3.metric('Decision', '\\U0001F6A8 FRAUD' if flag else '\\u2705 LEGITIMATE')

        if flag:
            st.error(
                '**Transaction FLAGGED as likely fraudulent** — '
                + f'P(fraud) = {prob:.1%} >= threshold {THRESHOLD:.3f}'
            )
        else:
            st.success(
                '**Transaction CLEARED as legitimate** — '
                + f'P(fraud) = {prob:.1%} < threshold {THRESHOLD:.3f}'
            )

        # ── Feature-deviation bar chart ───────────────────────────────────────
        # Shows how each feature deviates from typical legitimate transactions,
        # normalised by the fraud-vs-legit gap.  Red = toward fraud.
        if train_stats:
            st.subheader('What drove this decision?')
            st.caption(
                'Each bar shows how far this feature deviates from a typical '
                'legitimate transaction, scaled by the fraud-vs-legit gap. '
                'Red (right) = toward fraud. Blue (left) = toward legitimate.'
            )
            legit_m = train_stats.get('legit_mean', {})
            fraud_m = train_stats.get('fraud_mean', {})
            devs, feat_labels = [], []
            for feat in FEATURE_COLS:
                lm    = legit_m.get(feat, 0)
                fm    = fraud_m.get(feat, 0)
                scale = abs(fm - lm) if abs(fm - lm) > 1e-9 else 1.0
                devs.append((row[feat] - lm) / scale)
                feat_labels.append(feat)
            colors = ['#F44336' if d > 0 else '#2196F3' for d in devs]
            fig_dev = go.Figure(go.Bar(
                x=devs, y=feat_labels, orientation='h', marker_color=colors,
            ))
            fig_dev.update_layout(
                xaxis_title='Deviation (positive = toward fraud)',
                height=400,
                margin=dict(l=10, r=10, t=10, b=10),
            )
            st.plotly_chart(fig_dev, use_container_width=True)

        # ── Engineered feature values caption ─────────────────────────────────
        st.caption(
            'Derived features — '
            + 'errorBalanceOrig: ' + f'{err_orig:,.0f}'
            + '  |  errorBalanceDest: ' + f'{err_dest:,.0f}'
            + '  |  flag_orig_zero_after: ' + str(flag_orig_zero_after)
            + '  |  flag_dest_zero_before: ' + str(flag_dest_zero_before)
            + '  |  flag_dest_zero_both: ' + str(flag_dest_zero_both)
            + '  |  type_TRANSFER: ' + str(type_transfer)
        )

    # Pre-scored example table (always visible below the form)
    if demo_df is not None:
        st.divider()
        st.subheader('Pre-scored Example Transactions')
        st.caption(
            'Sample of 100 fraud + 400 legitimate transactions, pre-scored for demo. '
            'Not representative of the real 0.13% fraud rate.'
        )
        display_cols = [c for c in ['isFraud', 'amount', 'fraud_prob', 'flagged']
                        if c in demo_df.columns]
        st.dataframe(
            demo_df[display_cols]
            .sort_values('fraud_prob', ascending=False)
            .head(20),
            use_container_width=True,
        )

# ==========================================================================
# TAB 2 — Model Performance Dashboard
# ==========================================================================
with tab2:
    st.header('Model Performance Dashboard')

    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric('PR-AUC',    f"{metrics.get('pr_auc',    0):.3f}",
              help='Primary — not inflated by true negatives')
    c2.metric('ROC-AUC',   f"{metrics.get('roc_auc',   0):.3f}",
              help='Secondary — optimistic under class imbalance')
    c3.metric('F1',        f"{metrics.get('f1',        0):.3f}",
              help='Harmonic mean of precision and recall')
    c4.metric('Precision', f"{metrics.get('precision', 0):.3f}",
              help='Of flagged transactions, % actually fraudulent')
    c5.metric('Recall',    f"{metrics.get('recall',    0):.3f}",
              help='Of all real frauds, % caught at tuned threshold')

    st.divider()
    left_col, right_col = st.columns(2)

    with left_col:
        pr_path = 'reports/figures/pr_curves_comparison.png'
        if os.path.exists(pr_path):
            st.subheader('Precision-Recall Curve')
            st.image(pr_path)

        cm_found = False
        for cm_path in [
            'reports/figures/confusion_matrix_tuned_threshold.png',
            'reports/figures/confusion_matrices_before_after.png',
            'reports/figures/confusion_matrices_default_threshold.png',
        ]:
            if os.path.exists(cm_path):
                st.subheader('Confusion Matrix (threshold = ' + f'{THRESHOLD:.3f})')
                st.image(cm_path)
                cm_found = True
                break
        if not cm_found:
            st.info('Confusion matrix figure not found. Run Stages 9–10 to generate it.')

    with right_col:
        if shap_ranking and shap_mean_abs:
            st.subheader('Global Feature Importance (SHAP)')
            vals  = [shap_mean_abs.get(f, 0) for f in shap_ranking]
            fig_s = go.Figure(go.Bar(
                x=vals[::-1], y=shap_ranking[::-1],
                orientation='h', marker_color='#F44336',
            ))
            fig_s.update_layout(
                xaxis_title='Mean |SHAP Value|',
                height=420,
                margin=dict(l=10, r=10, t=10, b=10),
            )
            st.plotly_chart(fig_s, use_container_width=True)

        roc_path = 'reports/figures/roc_curves_comparison.png'
        if os.path.exists(roc_path):
            st.subheader('ROC Curve (secondary metric)')
            st.image(roc_path)

    st.divider()
    st.subheader('Why PR-AUC — Not Accuracy?')
    fraud_rate = metrics.get('fraud_rate_pct', 0.129)
    _fr_str    = str(round(fraud_rate, 3))
    _na_str    = str(round(100.0 - fraud_rate, 3))
    _base_str  = str(round(fraud_rate / 100, 4))
    st.markdown(
        "At PaySim's **" + _fr_str + '% fraud rate**, '
        + 'a model that always predicts *legitimate* for every transaction achieves **'
        + _na_str + '% accuracy** while catching **zero fraud**. '
        + 'This is the accuracy paradox — accuracy is dominated by the majority class '
        + 'and is blind to the minority you actually care about.\\n\\n'
        + '**PR-AUC** focuses exclusively on the positive (fraud) class. '
        + 'It cannot be inflated by correctly ignoring millions of legitimate '
        + 'transactions. Its random-baseline value equals the fraud rate itself '
        + '(~' + _base_str + '), '
        + 'so any lift above that baseline represents genuine predictive skill.'
    )

# ==========================================================================
# TAB 3 — Drift Monitor
# ==========================================================================
with tab3:
    st.header('Concept Drift Monitor')
    st.markdown(
        '**Concept drift** occurs when the statistical relationship between '
        'transaction features and the fraud label changes over time. '
        'As fraudsters adapt their tactics and customer behaviour shifts, '
        'a model trained on historical patterns can silently decay in accuracy '
        'without any visible error.\\n\\n'
        '_Drift was simulated by negating the strongest fraud-signal feature '
        'at the 70% mark of the temporally-ordered test stream._'
    )

    if drift_df is not None and len(drift_df) > 0:
        fig_drift = go.Figure()
        fig_drift.add_trace(go.Scatter(
            x=drift_df['idx'],
            y=drift_df['rolling_error'],
            mode='lines',
            name='Rolling Error (200-tx window)',
            line=dict(color='#2196F3', width=1.5),
        ))
        if inject_idx is not None:
            fig_drift.add_vline(
                x=inject_idx,
                line_dash='dot',
                line_color='#FF9800',
                annotation_text='Drift Injected',
                annotation_position='top left',
            )
        for ev in drift_events:
            fig_drift.add_vline(
                x=ev,
                line_dash='dash',
                line_color='#F44336',
                annotation_text='ADWIN Alert',
                annotation_position='top right',
            )
        fig_drift.update_layout(
            title='Model Error Rate Over Time — ADWIN Drift Detection',
            xaxis_title='Transaction Index (stream position)',
            yaxis_title='Rolling Error Rate',
            height=420,
            margin=dict(l=10, r=10, t=60, b=10),
        )
        st.plotly_chart(fig_drift, use_container_width=True)

        if drift_events:
            st.warning(
                '\\u26a0\\ufe0f **'
                + str(len(drift_events))
                + ' drift event(s) detected** at stream index: '
                + str(drift_events)
            )
            st.markdown('''**Production response playbook:**
1. **Alert** — log the detection timestamp and notify the ML team.
2. **Diagnose** — is this real concept shift, a data-pipeline bug, or a seasonal blip?
3. **Decide** — retrain only when drift is confirmed real, persistent, and fresh labelled data is available.''')
        else:
            st.info('No drift events detected in this replay.')

        with st.expander('\\U0001F4CA View raw error-stream data'):
            st.dataframe(
                drift_df[['idx', 'rolling_error']].rename(columns={
                    'idx': 'Transaction Index',
                    'rolling_error': 'Rolling Error Rate',
                }),
                use_container_width=True,
            )
    else:
        st.info(
            'Drift stream data not in bundle. '
            'Re-run fraud_detection_stage13_14.ipynb to regenerate.'
        )

    st.divider()
    st.subheader('About ADWIN (Adaptive Windowing)')
    st.markdown('''
**ADWIN** (Bifet & Gavaldà, SDM 2007) maintains a variable-length sliding window
over the model\'s binary error stream (1 = wrong prediction, 0 = correct).

At each new transaction it asks:
*"Can I split this window into an older sub-window and a newer one whose error
rates differ by more than statistical chance (a Hoeffding bound)?"*

- **Yes** — drift detected; old sub-window dropped, window shrinks to new regime.
- **No** — stable; window grows to accumulate more statistical power.

ADWIN self-calibrates: it shrinks after detecting change and grows during stability.
It provides formal guarantees on both false-alarm and missed-detection rates,
making it more principled than a simple rolling-average threshold rule.

The `delta` parameter controls sensitivity (lower = more sensitive).
This project uses `delta=0.002`.
''')
"""

with open(APP_PY_PATH, "w", encoding="utf-8") as fh:
    fh.write(APP_CODE)

size_kb = os.path.getsize(APP_PY_PATH) / 1024
print(f" app.py written → {APP_PY_PATH}  ({size_kb:.1f} KB)")
print()
print("  Feature set baked into Tab 1 (11 features):")
print("    Raw inputs  : amount, type_TRANSFER, oldbalanceOrg,")
print("                  newbalanceOrig, oldbalanceDest, newbalanceDest")
print("    Derived     : errorBalanceOrig, errorBalanceDest,")
print("                  flag_orig_zero_after, flag_dest_zero_before,")
print("                  flag_dest_zero_both")
print()
print("  Fraud-signature defaults on first load:")
print("    amount=55,000 | oldbalanceOrg=100,000 | newbalanceOrig=0")
print("    oldbalanceDest=0 | newbalanceDest=0 | type=TRANSFER")
print("    → errorBalanceDest=+55,000 (money never arrived at destination)")
print("    → errorBalanceOrig=-45,000 (origin lost more than transferred)")
print("    → all three flags = 1")
print()
print("  Launch: streamlit run app.py")

In [ ]:
# =============================================================================
# %% CELL 11 — Write requirements.txt to Disk
# =============================================================================


REQUIREMENTS = """# requirements.txt — Streamlit Cloud deployment dependencies
# Generated by fraud_detection_stage13_14.ipynb (Stage 14)
#
# These are the RUNTIME requirements for app.py only.
# Training dependencies (shap, imbalanced-learn, river, seaborn, matplotlib)
# are NOT needed here — the bundle contains all pre-computed artifacts.

streamlit>=1.30.0
pandas>=2.0.0
numpy>=1.24.0
xgboost>=1.7.0
scikit-learn>=1.2.0
plotly>=5.18.0
joblib>=1.3.0
pyarrow>=14.0.0
"""

with open(REQUIREMENTS_PATH, "w", encoding="utf-8") as fh:
    fh.write(REQUIREMENTS)

print(f" requirements.txt written → {REQUIREMENTS_PATH}")
print()
print("  Contents:")
print("  " + "-" * 50)
for line in REQUIREMENTS.strip().splitlines():
    print(f"  {line}")
print()
print("  Full training environment (for notebooks):")
print("  pip install streamlit pandas numpy xgboost scikit-learn plotly \\")
print("              joblib pyarrow shap imbalanced-learn river \\")
print("              seaborn matplotlib")

In [ ]:
# =============================================================================
# %% CELL 13 — Final Project Checkpoint
# =============================================================================


print("=" * 62)
print("  STAGE 14 COMPLETE — STREAMLIT APP EXPORTED")
print("=" * 62)
print()
print("  Files written by Stage 14:")
print(f"    app.py           → {APP_PY_PATH}")
print(f"    requirements.txt → {REQUIREMENTS_PATH}")
print()
print("  Quick launch:")
print("    streamlit run app.py")
print()
print("=" * 62)
print("  FULL PROJECT COMPLETE — ALL 15 STAGES")
print("=" * 62)
print()
print("  Stage 1–2   : Environment, imports, data loading & inspection")
print("  Stage 3     : Data cleaning (filter TRANSFER/CASH_OUT, drop leakage)")
print("  Stage 4     : Exploratory data analysis (EDA)")
print("  Stage 5     : Feature engineering (errorBalance*, zero flags, type flag)")
print("  Stage 6     : Time-aware train/test split")
print("  Stage 7     : Class imbalance (SMOTE + scale_pos_weight)")
print("  Stage 8     : Model training (LR baseline, XGB SPW, XGB SMOTE)")
print("  Stage 9     : Evaluation (PR-AUC, ROC-AUC, confusion matrices)")
print("  Stage 10    : Threshold tuning (cost function: FN=$500 / FP=$10)")
print("  Stage 11    : SHAP explainability (global + local waterfall plots)")
print("  Stage 12    : Concept drift monitoring (ADWIN on temporally-sorted stream)")
print("  Stage 13    : Deployment bundle assembly (models/streamlit_bundle.joblib)")
print("  Stage 14    : Streamlit app design & export (app.py + requirements.txt)")
print()
print("  Portfolio artifacts:")
print(f"    {BUNDLE_PATH}")
print(f"    {DEMO_DATA_PATH}")
print(f"    {APP_PY_PATH}")
print(f"    {REQUIREMENTS_PATH}")
print(f"    reports/figures/ (figures from Stages 9–12)")
print()
print("  Key portfolio talking points:")
print("    • PR-AUC as the primary metric (not accuracy — accuracy paradox)")
print("    • SMOTE-after-split rule (leakage prevention)")
print("    • errorBalance features as the primary fraud signal (domain-driven)")
print("    • Cost-aware threshold tuning ($500 FN / $10 FP cost matrix)")
print("    • SHAP for regulatory-grade per-prediction explainability")
print("    • ADWIN for continuous concept drift monitoring post-deployment")
print("    • Feature order enforced via the deployment bundle (silent-bug prevention)")
print("=" * 62)